In [ ]:
# Categories
# 1. Identifiers      => SUBJECT, SPELL_IDENTIFIER, ENCNTR_ID, ...
# 2. Dates / Raw text => ADMISSION_DATE, DISCHARGE_DATE, BIRTH_DATE, DEATH_DATE, ...
# 3. Target           =>
# 4. Features         =>

In [ ]:
# Missing
# IMD Decile -> Use an unknown category?
# Time since -> dslast, dsfirst,  ...
#  - Option 1: Fill missing with either -1 or 9999.
#  - Option 2: Pair with has prior event (hx) if using linear models.
#  - Tree models handle missing all right.

In [ ]:
# Encoding
# 1. Low cardinality (pd.get_dummies() or OneHotEcncoder.
#   MAIN_SPECIALTY_CODE_DESC
#   ETHNICITY_DES
#   GENDER_DESC

In [ ]:
# Splitting
# 1. Should we keep same patient in same fold? If evaluating temporal probably not?

In [8]:
# Libraries
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from icare_risk.utils.io import load_pkg_yaml

# Load data
schema = load_pkg_yaml("config/icare/schema.yaml")
DATA_DIR = Path(schema["data_root"])

# Load combined dataset from step 03
df = pd.read_parquet(DATA_DIR / "03_derived_analysis_dataset.parquet")
print(f"Loaded dataset shape: {df.shape}")

#
print(df.columns.tolist())

Loaded dataset shape: (10000, 186)
['SUBJECT', 'SPELL_IDENTIFIER', 'ENCNTR_ID', 'EPISODE_IDENTIFIER', 'ADMISSION_DATE', 'ADMISSION_TIME', 'AGE_AT_ADMISSION', 'DISCHARGE_DATE', 'MAIN_SPECIALTY_CODE', 'MAIN_SPECIALTY_CODE_DESC', 'INDEX_OF_MULTIPLE_DEPRIVATION_DECILE', 'index_admission', 'definitions', 'charlson_hx_mi', 'charlson_hx_chf', 'charlson_hx_pvd', 'charlson_hx_stroke', 'charlson_hx_dementia', 'charlson_hx_pulmonary', 'charlson_hx_rheum', 'charlson_hx_pud', 'charlson_hx_liver_mild', 'charlson_hx_diabetes_uncomp', 'charlson_hx_hemiplegia', 'charlson_hx_renal_mod_sev', 'charlson_hx_diabetes_comp', 'charlson_hx_cancer_solid', 'charlson_hx_leukemia', 'charlson_hx_lymphoma', 'charlson_hx_liver_mod_sev', 'charlson_hx_cancer_met', 'charlson_hx_aids', 'charlson_hx_hiv', 'kim_hx_prior_esbl', 'kim_hx_recent_hosp_1yr', 'kim_hx_nursing_home_resident', 'kim_hx_urinary_catheter', 'kim_hx_prior_antibiotics_90d', 'kim_hx_recent_antibiotics_90d', 'tumbarello_hx_prior_esbl', 'tumbarello_hx_recent_

In [31]:
# ==========================================
# 1. COLUMN GROUPING & METADATA DEFINITION
# ==========================================
# Administrative & Patient Identifiers (Drop from features)
ID_COLS = [
    "SUBJECT", "SPELL_IDENTIFIER", "ENCNTR_ID", "EPISODE_IDENTIFIER"
]

# Raw Timestamps & Dates (Drop from features to prevent leakage)
DATETIME_COLS = [
    "ADMISSION_DATE", "ADMISSION_TIME", "DISCHARGE_DATE",
    "BIRTH_DATE", "DEATH_DATE", "UPDATE_DT_TM",
    "closest_culture_date", "index_admission"
]

# Target & Label Columns (Isolated for modeling)
TARGET_COLS = [
    "esbl_status", "esbl_label_test", "esbl_binary"
]

# Categorical Features (Low/Medium cardinality descriptors)
CATEGORICAL_COLS = [
    "MAIN_SPECIALTY_CODE", "MAIN_SPECIALTY_CODE_DESC",
    "INDEX_OF_MULTIPLE_DEPRIVATION_DECILE",
    "ETHNICITY", "ETHNICITY_DES", "GENDER", "GENDER_DESC",
    "definitions", "qualifying_last_abx"
]

# Clinical Time-to-Event Gaps (_dslast, _dsfirst, days/time intervals)
TIME_COLS = [
    c for c in df.columns if any(s in c for s in ["_dslast", "_dsfirst", "time_to_culture", "days_since"])
]

# Everything else remaining (Binary history flags, counts, scores, vitals/rules)
# We can dynamically isolate these by subtracting all the above groups
excluded_cols = set(ID_COLS + DATETIME_COLS + TARGET_COLS + CATEGORICAL_COLS + TIME_COLS)
NUMERIC_FEATURE_COLS = [c for c in df.columns if c not in excluded_cols]

print(f"Total columns in dataframe: {df.shape[1]}")
print(f" - Identifiers: {len(ID_COLS)}")
print(f" - Datetime / Raw timestamps: {len(DATETIME_COLS)}")
print(f" - Target variables: {len(TARGET_COLS)}")
print(f" - Categorical features: {len(CATEGORICAL_COLS)}")
print(f" - Clinical time gaps (_dslast / _dsfirst): {len(TIME_COLS)}")
print(f" - Standard numeric features (Flags, counts, scores): {len(NUMERIC_FEATURE_COLS)}")

# Ensure target exists
TARGET = "esbl_binary"
df[TARGET] = (df["esbl_status"] == "ESBL").astype(int)

# Convert categorical columns explicitly
df_clean = df.copy()
for col in CATEGORICAL_COLS:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Missing").astype("category")

# Exclude administrative IDs, raw timestamps, and targets from feature matrix X
excluded_cols = set(ID_COLS + DATETIME_COLS + TARGET_COLS)
feature_cols = [c for c in df.columns if c not in excluded_cols]

X = df[feature_cols].copy()
y = df[TARGET].copy()

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")

Total columns in dataframe: 187
 - Identifiers: 4
 - Datetime / Raw timestamps: 8
 - Target variables: 3
 - Categorical features: 9
 - Clinical time gaps (_dslast / _dsfirst): 61
 - Standard numeric features (Flags, counts, scores): 102
Feature matrix X shape: (10000, 172)
Target vector y shape: (10000,)


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

# Custom transformer for time gaps to handle your clinical domain logic
class ClinicalTimeImputer(BaseEstimator, TransformerMixin):
    """
    Domain-aware imputer for clinical time-to-event gaps (_dslast, _dsfirst).
    Assigns a sentinel (-1.0) representing 'event never occurred / N/A'.
    """
    def __init__(self, sentinel: float = -1.0, suffixes: tuple = ("_dslast", "_dsfirst", "days_since", "time_to_culture")):
        self.sentinel = sentinel
        self.suffixes = suffixes
        self.time_cols_ = []

    def fit(self, X: pd.DataFrame, y=None):
        # Dynamically identify time columns from the input dataframe during fit
        if isinstance(X, pd.DataFrame):
            self.time_cols_ = [c for c in X.columns if any(c.endswith(s) or s in c for s in self.suffixes)]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X_out = X.copy()
        if not self.time_cols_ or not isinstance(X_out, pd.DataFrame):
            return X_out

        for col in self.time_cols_:
            if col in X_out.columns:
                # Fill missing lookback/time-to-event gaps with the sentinel (-1.0)
                X_out[col] = X_out[col].fillna(self.sentinel)

        return X_out

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return input_features
        return np.array(self.time_cols_)

# ==========================================
# DEFINE PREPROCESSORS (LR vs. XGBoost)
# ==========================================

# --- Preprocessor A: Linear Regression / MLP (Imputed + Scaled + One-Hot Encoded) ---
lr_preprocessor = ColumnTransformer(
    transformers=[
        ("time_gaps", Pipeline([
            ("imputer", ClinicalTimeImputer(sentinel=-1))
        ]), TIME_COLS),
        ("numerics", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), NUMERIC_FEATURE_COLS),
        ("categoricals", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), CATEGORICAL_COLS)
    ],
    remainder="drop"
)

# --- Preprocessor B: XGBoost / LightGBM (Minimal Imputation, No Scaling, Keeps Trees Efficient) ---
xgb_preprocessor = ColumnTransformer(
    transformers=[
        # Trees can handle missing time gaps natively via split directions, or we fill with sentinel if preferred
        ("time_gaps", Pipeline([
            ("imputer", ClinicalTimeImputer(sentinel=-1))
        ]), TIME_COLS),
        ("numerics", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), NUMERIC_FEATURE_COLS),
        ("categoricals", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), CATEGORICAL_COLS)
    ],
    remainder="drop"
)

In [25]:
import pandas as pd
import numpy as np

def inspect_preprocessor_output(preprocessor, X_sample, model_name="Model"):
    print(f"\n=============================================")
    print(f"🔍 INSPECTING PREPROCESSOR: {model_name}")
    print(f"=============================================")

    # Fit and transform
    X_trans = preprocessor.fit_transform(X_sample)

    # 1. Get real feature names (will now succeed without falling back to feat_0)
    raw_feature_names = preprocessor.get_feature_names_out()

    # 2. Clean up sklearn prefixes
    feature_names = [
        name.replace("time_gaps__", "")
            .replace("numerics__", "")
            .replace("categoricals__", "")
        for name in raw_feature_names
    ]

    # Convert to DataFrame with real names as column headers
    df_trans = pd.DataFrame(
        X_trans.toarray() if hasattr(X_trans, "toarray") else X_trans,
        columns=feature_names
    )

    print(f"Input shape: {X_sample.shape} ➔ Transformed shape: {df_trans.shape}")
    print(f"Total remaining NaNs: {df_trans.isna().sum().sum()}")
    print("-" * 50)

    # Separate Binary/OHE from Continuous/Time features
    binary_cols = []
    continuous_cols = []

    for col in df_trans.columns:
        unique_vals = df_trans[col].dropna().unique()
        if len(unique_vals) <= 2 and set(unique_vals).issubset({0.0, 1.0, 0, 1, -1.0}):
            binary_cols.append(col)
        else:
            continuous_cols.append(col)

    print(f"📊 Feature Breakdown:")
    print(f" • Continuous / Time / Score features: {len(continuous_cols)}")
    print(f" • Binary / One-Hot Encoded features: {len(binary_cols)}")

    if continuous_cols:
        print("\n--- 📈 Continuous / Time Features Summary ---")
        cont_summary = df_trans[continuous_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]
        cont_summary = cont_summary.rename(columns={'50%': 'median'})
        display(cont_summary.head(10))

    if binary_cols:
        print("\n--- 🔲 Binary / One-Hot Encoded Features Summary (Mean = Prevalence %) ---")
        bin_summary = df_trans[binary_cols].describe().T[['mean', 'min', 'max']]
        bin_summary = bin_summary.rename(columns={'mean': 'prevalence (proportion of 1s)'})
        display(bin_summary.head(10))

    return df_trans

In [26]:
# ==========================================
# 4. RUN & TEST BOTH PREPROCESSORS
# ==========================================
# Test Linear Regression pipeline
df_lr_ready = inspect_preprocessor_output(
    lr_preprocessor, X, model_name="Logistic Regression Pipeline")

# Test XGBoost pipeline
df_xgb_ready = inspect_preprocessor_output(
    xgb_preprocessor, X, model_name="XGBoost Pipeline")


🔍 INSPECTING PREPROCESSOR: Logistic Regression Pipeline
Input shape: (10000, 172) ➔ Transformed shape: (10000, 217)
Total remaining NaNs: 0
--------------------------------------------------
📊 Feature Breakdown:
 • Continuous / Time / Score features: 124
 • Binary / One-Hot Encoded features: 93

--- 📈 Continuous / Time Features Summary ---


,mean,std,min,median,max
charlson_hx_mi_dslast,1.524293,42.717440,-1.0,-1.0,1327.459028
charlson_hx_mi_dsfirst,1.524293,42.717440,-1.0,-1.0,1327.459028
charlson_hx_chf_dslast,2.423704,54.547590,-1.0,-1.0,1378.856250
charlson_hx_chf_dsfirst,2.459704,55.055093,-1.0,-1.0,1378.856250
charlson_hx_pvd_dslast,0.045119,25.370892,-1.0,-1.0,1181.495139
charlson_hx_pvd_dsfirst,0.045119,25.370892,-1.0,-1.0,1181.495139
charlson_hx_stroke_dslast,3.071855,54.385246,-1.0,-1.0,1364.440278
charlson_hx_stroke_dsfirst,3.097755,54.712979,-1.0,-1.0,1364.440278
charlson_hx_dementia_dslast,2.843614,53.214698,-1.0,-1.0,1392.031944
charlson_hx_dementia_dsfirst,2.843614,53.214698,-1.0,-1.0,1392.031944



--- 🔲 Binary / One-Hot Encoded Features Summary (Mean = Prevalence %) ---


,prevalence (proportion of 1s),min,max
days_since_last_discharge,-1.0,-1.0,-1.0
charlson_hx_hemiplegia,0.0,0.0,0.0
kim_hx_recent_hosp_1yr,0.0,0.0,0.0
kim_hx_nursing_home_resident,0.0,0.0,0.0
tumbarello_hx_recent_hosp_90d,0.0,0.0,0.0
jones_hx_prior_esbl_180d,0.0,0.0,0.0
jones_hx_chronic_dialysis,0.0,0.0,0.0
jones_hx_transfer_from_hospital,0.0,0.0,0.0
gavaghan_hx_prior_esbl_365d,0.0,0.0,0.0
gavaghan_hx_nursing_home_resident,0.0,0.0,0.0



🔍 INSPECTING PREPROCESSOR: XGBoost Pipeline
Input shape: (10000, 172) ➔ Transformed shape: (10000, 217)
Total remaining NaNs: 0
--------------------------------------------------
📊 Feature Breakdown:
 • Continuous / Time / Score features: 75
 • Binary / One-Hot Encoded features: 142

--- 📈 Continuous / Time Features Summary ---


,mean,std,min,median,max
charlson_hx_mi_dslast,1.524293,42.717440,-1.0,-1.0,1327.459028
charlson_hx_mi_dsfirst,1.524293,42.717440,-1.0,-1.0,1327.459028
charlson_hx_chf_dslast,2.423704,54.547590,-1.0,-1.0,1378.856250
charlson_hx_chf_dsfirst,2.459704,55.055093,-1.0,-1.0,1378.856250
charlson_hx_pvd_dslast,0.045119,25.370892,-1.0,-1.0,1181.495139
charlson_hx_pvd_dsfirst,0.045119,25.370892,-1.0,-1.0,1181.495139
charlson_hx_stroke_dslast,3.071855,54.385246,-1.0,-1.0,1364.440278
charlson_hx_stroke_dsfirst,3.097755,54.712979,-1.0,-1.0,1364.440278
charlson_hx_dementia_dslast,2.843614,53.214698,-1.0,-1.0,1392.031944
charlson_hx_dementia_dsfirst,2.843614,53.214698,-1.0,-1.0,1392.031944



--- 🔲 Binary / One-Hot Encoded Features Summary (Mean = Prevalence %) ---


,prevalence (proportion of 1s),min,max
days_since_last_discharge,-1.0000,-1.0,-1.0
charlson_hx_mi,0.0048,0.0,1.0
charlson_hx_chf,0.0055,0.0,1.0
charlson_hx_pvd,0.0025,0.0,1.0
charlson_hx_stroke,0.0086,0.0,1.0
charlson_hx_dementia,0.0077,0.0,1.0
charlson_hx_pulmonary,0.0079,0.0,1.0
charlson_hx_rheum,0.0045,0.0,1.0
charlson_hx_pud,0.0044,0.0,1.0
charlson_hx_liver_mild,0.0019,0.0,1.0


In [29]:
# Search for a specific feature
display(df_xgb_ready.filter(like='GENDER').describe().T)

,count,mean,std,min,25%,50%,75%,max
GENDER_F,10000.0,0.2527,0.434582,0.0,0.0,0.0,1.0,1.0
GENDER_I,10000.0,0.2504,0.433265,0.0,0.0,0.0,1.0,1.0
GENDER_M,10000.0,0.2535,0.435036,0.0,0.0,0.0,1.0,1.0
GENDER_U,10000.0,0.2434,0.429156,0.0,0.0,0.0,0.0,1.0
GENDER_DESC_Female,10000.0,0.2527,0.434582,0.0,0.0,0.0,1.0,1.0
GENDER_DESC_Indeterminate,10000.0,0.2504,0.433265,0.0,0.0,0.0,1.0,1.0
GENDER_DESC_Male,10000.0,0.2535,0.435036,0.0,0.0,0.0,1.0,1.0
GENDER_DESC_Unknown,10000.0,0.2434,0.429156,0.0,0.0,0.0,0.0,1.0


In [33]:
# ==========================================
# 1. LOAD DATA & CLEAN TYPE CASTING
# ==========================================
import pandas as pd
import numpy as np
from pathlib import Path
from tableone import TableOne
from icare_risk.utils.io import load_pkg_yaml

# Load project configuration and schema paths
schema = load_pkg_yaml("config/icare/schema.yaml")
DATA_DIR = Path(schema["data_root"])

# Load combined dataset from step 03
df_master = pd.read_parquet(DATA_DIR / "03_derived_analysis_dataset.parquet")
print(f"Loaded dataset shape: {df_master.shape}")

# Ensure target exists for modeling and table grouping
TARGET = "esbl_binary"
df_master[TARGET] = (df_master["esbl_status"] == "ESBL").astype(int)
df_master["esbl_label"] = np.where(df_master[TARGET] == 1, "Positive", "Negative")


# ==========================================
# 2. FEATURE GROUPING & DATATYPE CONVERSION
# ==========================================
ID_COLS = ["SUBJECT", "SPELL_IDENTIFIER", "ENCNTR_ID", "EPISODE_IDENTIFIER"]
DATETIME_COLS = [
    "ADMISSION_DATE", "ADMISSION_TIME", "DISCHARGE_DATE",
    "BIRTH_DATE", "DEATH_DATE", "UPDATE_DT_TM",
    "closest_culture_date", "index_admission"
]
TARGET_COLS = ["esbl_status", "esbl_label_test", TARGET, "esbl_label"]

# Explicitly define core categorical columns
CATEGORICAL_COLS = [
    "MAIN_SPECIALTY_CODE", "MAIN_SPECIALTY_CODE_DESC",
    "INDEX_OF_MULTIPLE_DEPRIVATION_DECILE",
    "ETHNICITY", "ETHNICITY_DES", "GENDER", "GENDER_DESC",
    "definitions", "qualifying_last_abx"
]

# Identify clinical time gaps
TIME_COLS = [
    c for c in df_master.columns if any(s in c for s in ["_dslast", "_dsfirst", "time_to_culture", "days_since"])
]

# Convert categorical columns explicitly in pandas to 'category' or clean strings
df_clean = df_master.copy()
for col in CATEGORICAL_COLS:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Missing").astype("category")

# Isolate standard numeric features (flags, counts, scores) excluding IDs, dates, targets, categoricals, and time gaps
excluded_cols = set(ID_COLS + DATETIME_COLS + TARGET_COLS + CATEGORICAL_COLS + TIME_COLS)
NUMERIC_FEATURE_COLS = [c for c in df_clean.columns if c not in excluded_cols]

# Build feature matrix X for scikit-learn
feature_cols = [c for c in df_clean.columns if c not in set(ID_COLS + DATETIME_COLS + TARGET_COLS)]
X = df_clean[feature_cols].copy()
y = df_clean[TARGET].copy()

print(f"✅ Data preparation complete:")
print(f" - Categorical features: {len(CATEGORICAL_COLS)}")
print(f" - Clinical time gaps: {len(TIME_COLS)}")
print(f" - Standard numeric features: {len(NUMERIC_FEATURE_COLS)}")


# ==========================================
# 3. DEFINE PREPROCESSORS (LR vs. XGBoost)
# ==========================================
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

class ClinicalTimeImputer(BaseEstimator, TransformerMixin):
    """
    Domain-aware imputer for clinical time gaps (_dslast, _dsfirst).
    Assigns a sentinel (-1.0) representing 'event never occurred / N/A'.
    """
    def __init__(self, sentinel: float = -1.0, suffixes: tuple = ("_dslast", "_dsfirst", "days_since", "time_to_culture")):
        self.sentinel = sentinel
        self.suffixes = suffixes
        self.time_cols_ = []

    def fit(self, X: pd.DataFrame, y=None):
        if isinstance(X, pd.DataFrame):
            self.time_cols_ = [c for c in X.columns if any(c.endswith(s) or s in c for s in self.suffixes)]
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X_out = X.copy()
        if not self.time_cols_ or not isinstance(X_out, pd.DataFrame):
            return X_out
        for col in self.time_cols_:
            if col in X_out.columns:
                X_out[col] = X_out[col].fillna(self.sentinel)
        return X_out

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return input_features
        return np.array(self.time_cols_)

# Preprocessor A: Linear Regression / MLP (Scaled + One-Hot)
lr_preprocessor = ColumnTransformer(
    transformers=[
        ("time_gaps", Pipeline([("imputer", ClinicalTimeImputer(sentinel=-1))]), TIME_COLS),
        ("numerics", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), NUMERIC_FEATURE_COLS),
        ("categoricals", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Missing")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL_COLS)
    ],
    remainder="drop"
)

# Preprocessor B: XGBoost / LightGBM (Unscaled + One-Hot)
xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("time_gaps", Pipeline([("imputer", ClinicalTimeImputer(sentinel=-1))]), TIME_COLS),
        ("numerics", Pipeline([("imputer", SimpleImputer(strategy="median"))]), NUMERIC_FEATURE_COLS),
        ("categoricals", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Missing")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL_COLS)
    ],
    remainder="drop"
)


# ==========================================
# 4. REUSABLE INSPECTION FUNCTION
# ==========================================
def inspect_preprocessor_output(preprocessor, X_sample, model_name="Model"):
    print(f"\n=============================================")
    print(f"🔍 INSPECTING PREPROCESSOR: {model_name}")
    print(f"=============================================")

    X_trans = preprocessor.fit_transform(X_sample)
    raw_feature_names = preprocessor.get_feature_names_out()

    feature_names = [
        name.replace("time_gaps__", "")
            .replace("numerics__", "")
            .replace("categoricals__", "")
        for name in raw_feature_names
    ]

    df_trans = pd.DataFrame(
        X_trans.toarray() if hasattr(X_trans, "toarray") else X_trans,
        columns=feature_names
    )

    print(f"Input shape: {X_sample.shape} ➔ Transformed shape: {df_trans.shape}")
    print(f"Total remaining NaNs: {df_trans.isna().sum().sum()}")
    print("-" * 50)

    return df_trans

# Run & test pipelines
df_lr_ready = inspect_preprocessor_output(lr_preprocessor, X, model_name="Logistic Regression Pipeline")
df_xgb_ready = inspect_preprocessor_output(xgb_preprocessor, X, model_name="XGBoost Pipeline")


# ==========================================
# 5. GENERATE CLINICAL TABLE ONE SUMMARY
# ==========================================
print("\n=============================================")
print("📋 GENERATING EPIDEMIOLOGICAL TABLE ONE")
print("=============================================")

tableone_cols = [c for c in feature_cols if c in CATEGORICAL_COLS or c in NUMERIC_FEATURE_COLS or c == "AGE_AT_ADMISSION"]
nonnormal_cols = [c for c in NUMERIC_FEATURE_COLS if c != "AGE_AT_ADMISSION"]

mytable = TableOne(
    data=df_clean,
    columns=tableone_cols,
    categorical=[c for c in CATEGORICAL_COLS if c in df_clean.columns],
    nonnormal=nonnormal_cols,
    groupby="esbl_label",
    overall=True,
    pval=False
)

# Display summary in notebook
print(mytable.tabulate(tablefmt="github"))

# Optionally export to CSV
output_csv = DATA_DIR / "table1_demographics.csv"
mytable.to_csv(output_csv)
print(f"\n📁 Table 1 saved successfully to: {output_csv}")

Loaded dataset shape: (10000, 186)
✅ Data preparation complete:
 - Categorical features: 9
 - Clinical time gaps: 61
 - Standard numeric features: 102

🔍 INSPECTING PREPROCESSOR: Logistic Regression Pipeline
Input shape: (10000, 172) ➔ Transformed shape: (10000, 217)
Total remaining NaNs: 0
--------------------------------------------------

🔍 INSPECTING PREPROCESSOR: XGBoost Pipeline
Input shape: (10000, 172) ➔ Transformed shape: (10000, 217)
Total remaining NaNs: 0
--------------------------------------------------

📋 GENERATING EPIDEMIOLOGICAL TABLE ONE
|                                                       |                                      |   Missing | Overall       | Negative      | Positive      |
|-------------------------------------------------------|--------------------------------------|-----------|---------------|---------------|---------------|
| n                                                     |                                      |           | 10000         